## Q1 — Active Revenue

**Answer:**
- Active subscriptions: 2,340
- Total monthly recurring revenue: ₹7,84,260

As of 30 June 2024, there were 2,340 active subscriptions generating a total monthly recurring revenue of ₹7,84,260.

In [ ]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

pd.read_sql(query, engine)

## Q2 — Signup Momentum

**Answer:**

* January: 350 signups
* February: 400 signups
* March: 500 signups
* April: 550 signups
* May: 600 signups
* June: 600 signups

**Highest signups:** May and June, with 600 signups each.

The number of new user signups increased steadily from January through June 2024. May and June recorded the highest number of signups, with 600 new users each.


In [ ]:
SELECT
MONTH(signup_date) AS month,
COUNT(*) AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
  AND MONTH(signup_date) BETWEEN 1 AND 6
GROUP BY MONTH(signup_date)
ORDER BY MONTH(signup_date);

## Q3 — Device Analytics

**Answer:**

* **Laptop:** 15,105 sessions, 453,434 total watch minutes, 30.02 average watch minutes/session, 60.51% completion rate.
* **Mobile:** 50,172 sessions, 1,504,355 total watch minutes, 29.98 average watch minutes/session, 60.24% completion rate.
* **Tablet:** 7,091 sessions, 210,733 total watch minutes, 29.72 average watch minutes/session, 59.79% completion rate.
* **TV:** 27,981 sessions, 840,595 total watch minutes, 30.04 average watch minutes/session, 59.98% completion rate.

Mobile has the highest number of sessions and total watch minutes. Laptop has the highest completion rate at 60.51%, while TV has the highest average watch time per session at 30.04 minutes.


In [ ]:
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;

## Q4 — Rating Distribution

**Answer:**

* 1 star: 234 ratings (4.68%)
* 2 stars: 352 ratings (7.04%)
* 3 stars: 847 ratings (16.94%)
* 4 stars: 1,781 ratings (35.62%)
* 5 stars: 1,786 ratings (35.72%)

**Percentage of 4 or 5 star ratings: 71.34%**

The majority of users gave positive ratings, with 71.34% of all ratings being 4 or 5 stars. The most common rating was 5 stars, accounting for 35.72% of all ratings.


In [ ]:
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings),
        2
    ) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;

SELECT
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS four_or_five_star_percentage
FROM ratings;

## Q5 — Originals vs Acquired

**Answer:**

| Content Type        | Number of Shows | Average IMDb Rating | Average Release Year |
| ------------------- | --------------: | ------------------: | -------------------: |
| BingePlay Originals |              30 |                7.92 |              2020.37 |
| Acquired Content    |              70 |                6.63 |              2020.73 |

**Interpretation:**
BingePlay Originals perform better than acquired content based on average IMDb rating. The average rating of Originals is **1.29 points higher** than that of acquired content.


In [ ]:
SELECT
    CASE
        WHEN is_original = 1 THEN 'BingePlay Originals'
        ELSE 'Acquired Content'
    END AS content_type,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS average_imdb_rating,
    ROUND(AVG(release_year), 2) AS average_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;

## Q6 — Binge Day Detection

**Answer:**

* Total binge days: **414**
* User with the most binge days: **U02956**
* Number of binge days for U02956: **8**

A binge day occurs when the same user watches the same show at least five times on the same calendar date. In Q2 2024, there were 414 such binge days, and user U02956 had the highest count with 8 binge days.


In [ ]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date
    FROM watch_sessions
    WHERE session_date BETWEEN '2024-04-01' AND '2024-06-30'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
user_binge_counts AS (
    SELECT
        user_id,
        COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
),
top_user AS (
    SELECT user_id, binge_days
    FROM user_binge_counts
    ORDER BY binge_days DESC
    LIMIT 1
)
SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    top_user.user_id AS top_user_id,
    top_user.binge_days AS top_user_binge_days
FROM top_user;
"""

pd.read_sql(query, engine)

## Q7 — Q1 Signups Who Never Watched

**Answer:**

* Total Q1 signups: **1,250**
* Q1 signups who never watched anything: **226 users**

**Interpretation:**
Out of 1,250 users who signed up during Q1 2024, 226 users had never watched a single session. The query uses a LEFT JOIN with IS NULL to safely identify users without any watch sessions.


In [ ]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE
            WHEN ws.session_id IS NULL THEN 1
            ELSE 0
        END
    ) AS never_watched_q1_users
FROM users u
LEFT JOIN watch_sessions ws
    ON u.user_id = ws.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-04-01';
"""

pd.read_sql(query, engine)

## Q8 — The Over-Paying Premium/Family Users

**Answer: 6 users**

There are 6 current Premium/Family users whose entire watch history consists only of shows available on the Basic plan. These users may be suitable targets for a downgrade offer.


In [ ]:
query = """
WITH current_subscriptions AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND start_date <= '2024-06-30'
      AND (end_date IS NULL OR end_date > '2024-06-30')
),
premium_family_users AS (
    SELECT user_id
    FROM current_subscriptions
    WHERE rn = 1
      AND plan IN ('Premium', 'Family')
)
SELECT COUNT(*) AS over_paying_users
FROM premium_family_users p
WHERE EXISTS (
    SELECT 1
    FROM watch_sessions ws
    WHERE ws.user_id = p.user_id
)
AND NOT EXISTS (
    SELECT 1
    FROM watch_sessions ws
    JOIN shows s
        ON ws.show_id = s.show_id
    WHERE ws.user_id = p.user_id
      AND s.min_plan IN ('Premium', 'Family')
);
"""

pd.read_sql(query, engine)

## Q9 — Upgrade Success Cohort

**Answer:**

* Upgrade success users: **55**
* Average days from signup to first upgrade: **64.96 days**

**Interpretation:**
Among users who signed up in January 2024, started with the Basic plan, later upgraded to Premium or Family, and were still active as of 30 June 2024, there were 55 successful upgrade users. On average, they took 64.96 days from signup to their first upgrade.


In [ ]:
query = """
WITH january_users AS (
    SELECT
        user_id,
        signup_date
    FROM users
    WHERE signup_date >= '2024-01-01'
      AND signup_date < '2024-02-01'
),
ranked_subscriptions AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
    JOIN january_users u
        ON s.user_id = u.user_id
),
basic_starters AS (
    SELECT user_id
    FROM ranked_subscriptions
    WHERE rn = 1
      AND plan = 'Basic'
),
first_upgrade AS (
    SELECT
        rs.user_id,
        MIN(rs.start_date) AS first_upgrade_date
    FROM ranked_subscriptions rs
    JOIN basic_starters bs
        ON rs.user_id = bs.user_id
    WHERE rs.plan IN ('Premium', 'Family')
      AND rs.start_date > (
          SELECT MIN(rs2.start_date)
          FROM ranked_subscriptions rs2
          WHERE rs2.user_id = rs.user_id
            AND rs2.rn = 1
      )
    GROUP BY rs.user_id
),
active_users AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND start_date <= '2024-06-30'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT
    COUNT(*) AS upgrade_success_users,
    ROUND(
        AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)),
        2
    ) AS avg_days_to_first_upgrade
FROM first_upgrade fu
JOIN january_users u
    ON fu.user_id = u.user_id
JOIN active_users au
    ON fu.user_id = au.user_id;
"""

pd.read_sql(query, engine)

## Q10 — Cliffhanger Comebacks

**Answer:**

* Total cliffhanger comeback events: **4,345**
* Show with the most comebacks: **S088**
* Title: **Rayalaseema Raga**

**Interpretation:**
There were 4,345 unique cliffhanger comeback events. Show S088, *Rayalaseema Raga*, generated the highest number of comeback events.


In [ ]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        ws1.user_id,
        ws1.show_id,
        ws1.session_date AS incomplete_date
    FROM watch_sessions ws1
    JOIN watch_sessions ws2
        ON ws1.user_id = ws2.user_id
        AND ws1.show_id = ws2.show_id
        AND ws2.session_date BETWEEN
            DATE_ADD(ws1.session_date, INTERVAL 1 DAY)
            AND DATE_ADD(ws1.session_date, INTERVAL 7 DAY)
    WHERE ws1.completed = 0
      AND ws1.user_id IS NOT NULL
)
SELECT
    COUNT(*) AS total_comeback_events,
    (
        SELECT ce.show_id
        FROM comeback_events ce
        GROUP BY ce.show_id
        ORDER BY COUNT(*) DESC, ce.show_id
        LIMIT 1
    ) AS top_show_id,
    (
        SELECT s.title
        FROM shows s
        JOIN (
            SELECT show_id
            FROM comeback_events
            GROUP BY show_id
            ORDER BY COUNT(*) DESC, show_id
            LIMIT 1
        ) top_show
        ON s.show_id = top_show.show_id
    ) AS top_show_title
FROM comeback_events;
"""

pd.read_sql(query, engine)

## Q11 — Consecutive-Week Engagement

**Answer:**

* Users with a streak of 4+ consecutive weeks: **1,675**
* Longest streak: **26 weeks**
* One user with the longest streak: **U00213**

**Interpretation:**
A total of 1,675 users watched at least one session in four or more consecutive calendar weeks. The longest engagement streak was 26 consecutive weeks, achieved by user U00213.


In [ ]:
query = """
WITH weekly_activity AS (
    SELECT DISTINCT
        user_id,
        YEARWEEK(session_date, 3) AS week_key
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        week_key,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_key
        ) AS rn
    FROM weekly_activity
),
streaks AS (
    SELECT
        user_id,
        week_key - rn AS streak_group
    FROM numbered_weeks
),
streak_lengths AS (
    SELECT
        user_id,
        streak_group,
        COUNT(*) AS streak_length
    FROM streaks
    GROUP BY user_id, streak_group
),
users_with_4plus AS (
    SELECT DISTINCT user_id
    FROM streak_lengths
    WHERE streak_length >= 4
),
longest AS (
    SELECT
        user_id,
        streak_length
    FROM streak_lengths
    ORDER BY streak_length DESC, user_id
    LIMIT 1
)
SELECT
    (SELECT COUNT(*) FROM users_with_4plus) AS users_with_4plus_streak,
    longest.streak_length AS longest_streak_weeks,
    longest.user_id AS longest_streak_user
FROM longest;
"""

pd.read_sql(query, engine)

# Q12
522 users were identified as churn signals because their total watch time in June 2024 decreased by 50% or more compared with May 2024. Users with zero watch time in June were also included.

In [ ]:
WITH monthly_watch AS (
    SELECT
        user_id,
        DATE_FORMAT(session_date, '%Y-%m') AS month,
        SUM(watch_minutes) AS month_mins
    FROM watch_sessions
    WHERE session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id, DATE_FORMAT(session_date, '%Y-%m')
),

user_months AS (
    SELECT
        user_id,

        SUM(
            CASE
                WHEN month = '2024-05' THEN month_mins
                ELSE 0
            END
        ) AS may_minutes,

        SUM(
            CASE
                WHEN month = '2024-06' THEN month_mins
                ELSE 0
            END
        ) AS june_minutes

    FROM monthly_watch
    GROUP BY user_id
),

churn_signals AS (
    SELECT
        user_id,
        may_minutes,
        june_minutes,
        ROUND(
            ((may_minutes - june_minutes) / may_minutes) * 100,
            2
        ) AS drop_percentage
    FROM user_months
    WHERE may_minutes > 0
      AND june_minutes <= may_minutes * 0.5
)

SELECT
    cs.user_id,
    u.name,
    cs.may_minutes AS total_may_2024_watch_minutes,
    cs.june_minutes AS total_june_2024_watch_minutes,
    cs.drop_percentage
FROM churn_signals cs
JOIN users u
    ON cs.user_id = u.user_id
ORDER BY cs.drop_percentage DESC;